# T6. 공공데이터 품질 관제판

이 과정의 프로젝트가 쓰는 공공 데이터셋 13개(서울 열린데이터광장 10 · 공공데이터포털 3)의 화면에서 최종 갱신일 · 갱신 주기 · 파일 목록 · 설명을 모아 「갱신이 멈춘 데이터」를 잡고, 받아 둔 스냅샷으로 형식 지문(인코딩 · 열 · 행 수)과 빈 값 · 행 수 변화를 재 알림 목록과 Streamlit 관제판으로 보여 줍니다. 데이터셋 설명 · 변경 안내 · 자주 하는 질문을 Qdrant 에 넣어 「이 데이터 왜 끊겼나」에 근거 문장으로 답합니다.

- 단계: 팀 프로젝트 · 정형 + 비정형 · 예상 2~3주 규모(혼자 진행 · 노트북 한 번 따라하기 3~4시간)
- 실행 환경: Google Colab(브라우저만 있으면 됩니다)
- 데이터: 서울 열린데이터광장 · 공공데이터포털 — 데이터셋 13개의 화면 메타데이터(제목 · 공개일 · 갱신일 · 갱신 주기 · 파일 목록 · 설명 · 자주 하는 질문), 2026-10-29 20:49 수집한 스냅샷
- 데이터: 서울 열린데이터광장 — 서울시 일별 평균 대기오염도 정보(OA-2218) 2023 · 2024 · 2025년 파일, 공공누리 1유형(출처표시)
- 데이터: 서울 열린데이터광장 — 공공자전거 대여소 정보(OA-13252) 24.12 · 25.6 · 25.12 · 26.6월 기준 파일, 공공누리 1유형
- 데이터: 서울 열린데이터광장 — 서울시 부동산 실거래가 정보(OA-21275) 계약일 2026년 6~9월분, 공공누리 1유형
- 데이터: 서울 열린데이터광장 — 행정동 단위 서울 생활인구(내국인)(OA-14991) 2026년 6월분, 공공누리 1유형
- 데이터: 공공데이터포털 — 국민권익위원회_고충민원 분야별 접수현황(15131929), 이용허락범위 제한 없음

**빈칸 노트북** — 코드 속 `___①___` 자리 16곳을 채웁니다. 줄 끝 주석이 힌트입니다.
막히면 [따라하기 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/T6/T6_guide.ipynb)의 같은 단계를 보세요. 정답은 맨 아래에 있습니다.

데이터 파이프라인은 코드가 멀쩡해도 깨집니다. **원천 데이터가 조용히 바뀌기 때문입니다.** 갱신이 멈추고, 인코딩이 바뀌고, 최근 달의 행이 덜 차 있고, 코드표가 바뀝니다.
이 과정의 프로젝트들도 그런 일을 겪었습니다 — P3 · T5 는 생활인구의 행정동 코드가 2016년 기준이라 상가정보와 21곳이 어긋났고, 생활인구는 2026년 7월분을 끝으로 행정동 단위 생산이 끝났습니다.

이 프로젝트는 그런 변화를 **먼저 알아채는 관제판**을 만듭니다. 대상은 이 과정의 프로젝트(M1~T5)가 쓰는 공공 데이터셋 13개입니다.

```text
데이터셋 화면 13개 ─(메타 수집, 요청 간격 1.5초)─▶ 메타 표 ─▶ 규칙 ① 신선도(갱신 주기 대비 경과일 · 마지막 파일의 달)
받아 둔 스냅샷 파일 ─────────────────────────────▶ 규칙 ② 형식 지문(인코딩 · 따옴표 · 열 · 행 수)
                                                 └▶ 규칙 ③ 품질 지표(빈 값 비율 · 기간별 행 수 변화)
                     세 규칙의 위반 ─▶ 알림 목록 ─▶ matplotlib 그림 · Streamlit 관제판
설명 · 변경 안내 · 자주 하는 질문 ─▶ 임베딩 · Qdrant ─▶ 「이 데이터 왜 끊겼나」 근거 찾기
Airflow DAG(매일) 가 수집 → 규칙 → 색인을 차례로 부릅니다.
```

## 네 갈래로 나누는 법

| 역할 | 맡는 단계 |
|---|---|
| 수집 · 스트리밍 | 2 화면 하나 뜯어보기 · 3 13개 수집과 스냅샷 |
| 레이크 · 품질 | 4 메타 표 · 5 신선도 규칙 · 6 형식 지문 · 7 품질 지표 |
| RAG · 서비스 | 8 알림 · 관제판 · 9 「왜 끊겼나」 검색 |
| 오케스트레이션 | 10 Airflow DAG · 11 더 해 보기 |

다 끝내면 다음을 말할 수 있습니다.

- 「갱신 주기가 매일인데 85일째 그대로」 같은 신호를 화면 메타데이터만으로 어떻게 잡는지, 그리고 그 규칙이 어디서 헛알림을 내는지
- 열 이름이 같아도 파일 형식이 바뀐 것(인코딩 · 따옴표)을 지문으로 잡는 방법
- 빈 값이 많은 열 가운데 「원래 비어야 정상인 열」을 어떻게 구별하는지
- 알림이 뜬 데이터의 **이유**를 변경 공지에서 찾아 붙이는 방법

> **자주 겪는 오류** — 2 · 3단계는 실제 사이트에 요청을 보냅니다. 화면 13개를 1.5초 간격으로 한 번씩만 받습니다 — 반복 실행할 때는 3단계의 `LIVE = False` 로 바꿔 사이트에 부담을 주지 마세요.
> 실시간으로 바뀌는 메타데이터는 형태(열 · 날짜 모양)로 확인하고, 숫자는 **2026-10-29 에 수집해 둔 스냅샷**으로 고정합니다.

## 1단계 · 준비 — 라이브러리와 스냅샷

Colab 에는 pandas · DuckDB · requests · BeautifulSoup 이 이미 있습니다. 7주차의 qdrant-client · sentence-transformers 와 8주차의 streamlit 만 없으면 설치합니다.

받아 둘 파일은 셋으로 나뉩니다.

- **메타 스냅샷** `t6_meta_20261029.json` — 3단계 수집 코드를 2026-10-29 20:49 에 돌려 저장한 것입니다. 데이터셋 13개의 화면에 공개된 제목 · 날짜 · 갱신 주기 · 파일 목록 · 설명 · 자주 하는 질문만 담았습니다.
- **형식 비교용 파일** — 서울 열린데이터광장 「서울시 일별 평균 대기오염도 정보」(OA-2218)의 2023 · 2024년 파일(원본 바이트 그대로 gzip)과 M2 의 2025년 파일, 「공공자전거 대여소 정보」(OA-13252)의 네 시점 파일(P9 과 같은 파일). 모두 공공누리 1유형(출처표시)입니다.
- **품질 지표용 스냅샷** — 앞 프로젝트들이 받아 둔 실거래가(P1 · OA-21275, 공공누리 1유형) · 생활인구(P3 · OA-14991, 공공누리 1유형) · 고충민원 통계(T2 · 공공데이터포털 15131929, 이용허락 제한 없음).

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("qdrant_client", "qdrant-client")
need("sentence_transformers", "sentence-transformers")
need("streamlit")

import os, re, io, json, gzip, time, shutil, urllib.request
from datetime import date, datetime
import requests
import pandas as pd
import duckdb
from bs4 import BeautifulSoup

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DATA = "https://raw.githubusercontent.com/aebonlee/materials/main/data-eng/projects/data/"

def fetch(name):
    """프로젝트 데이터를 ./data 폴더에 한 번만 받아 둡니다."""
    os.makedirs("data", exist_ok=True)
    dst = os.path.join("data", name)
    if not os.path.exists(dst):
        local = os.environ.get("DE_DATA_LOCAL")          # 오프라인 실습용 — 자료 폴더가 내 컴퓨터에 있을 때
        if local:
            shutil.copy(os.path.join(local, name), dst)
        else:
            urllib.request.urlretrieve(DATA + name, dst)
    return dst

FILES = ["t6_meta_20261029.json",
         "t6_seoul_air_daily_2023.csv.gz", "t6_seoul_air_daily_2024.csv.gz", "seoul_air_daily_2025.csv.gz",
         "p9_bike_station_202412.xlsx", "p9_bike_station_202506.xlsx", "p9_bike_station_202512.xlsx", "p9_bike_station_202606.xlsx",
         "seoul_realestate_trade_202606_09.csv.gz", "seoul_living_pop_dong_202606.csv.gz", "acrc_complaints_by_field_2015_2025.csv"]
for f in FILES:
    fetch(f)
print(len(os.listdir("data")), "개 파일 ·", round(sum(os.path.getsize(os.path.join("data", f)) for f in FILES) / 1e6, 1), "MB")

> **확인 포인트** — 파일 11개, 합쳐 약 5MB 입니다.

In [ ]:
assert all(os.path.exists(os.path.join("data", f)) for f in FILES)

## 2단계 · 데이터셋 화면 하나 뜯어보기 (수집 · 스트리밍)

서울 열린데이터광장의 데이터셋 화면(`/dataList/<OA 번호>/<유형>/1/datasetView.do`)은 HTML 표에 메타데이터를 담고 있습니다. 「데이터 정보」 표는 `<th>항목</th><td>값</td>` 짝이고, 파일 목록은 표의 한 행이 파일 하나입니다(11장 크롤러).
생활인구(OA-14991) 화면 하나로 구조를 확인합니다. 일부 기관 사이트는 기본 User-Agent 를 막으므로 브라우저 이름을 붙입니다.

In [ ]:
UA = {"User-Agent": "Mozilla/5.0 (Macintosh)"}

def clean(s):
    return re.sub(r"\s+", " ", s or "").strip()

url = "https://data.seoul.go.kr/dataList/OA-14991/S/1/datasetView.do"
soup = BeautifulSoup(requests.get(url, headers=UA, timeout=30).text, "lxml")

info = {}
for th in soup.select("___①___"):   # ① 표의 머리 칸 태그
    td = th.___②___("td")   # ② 바로 옆 형제 태그
    if td:
        info.setdefault(clean(th.get_text()), clean(td.get_text(" ")))
title = soup.find("meta", attrs={"name": "title"})["content"]
print(title)
{k: info.get(k) for k in ["공개일자", "데이터 갱신일", "갱신주기", "메타정보 수정일", "이용허락범위"]}

> **확인 포인트** — 제목 `행정동 단위 서울 생활인구(내국인)` 과 다섯 항목이 찍힙니다. 날짜는 `2026.08.05.` 처럼 **점으로 끝나는 모양**입니다. 값은 사이트가 바뀌면 달라지므로 모양만 확인합니다.

In [ ]:
assert title == "행정동 단위 서울 생활인구(내국인)"
assert re.fullmatch(r"\d{4}\.\d\d\.\d\d\.", info["데이터 갱신일"]) and "공공누리" in info["이용허락범위"]

일부러 없는 데이터셋 번호로 받아 보세요. HTTP 는 200 으로 성공하는데, 데이터셋 화면 대신 **열린데이터광장 첫 화면**이 돌아옵니다 — 상태 코드만 보고 「잘 받았다」고 판단하면 안 되는 이유입니다(T1 3단계와 같은 함정).
그래서 「데이터 갱신일」 칸이 있는지로 데이터셋 화면이 맞는지 확인합니다.

In [ ]:
r = requests.get("https://data.seoul.go.kr/dataList/OA-99999999/S/1/datasetView.do", headers=UA, timeout=30)
s2 = BeautifulSoup(r.text, "lxml")
has_date = any(clean(th.get_text()) == "데이터 갱신일" for th in s2.select("th"))
print("HTTP", r.status_code, "· 제목:", s2.find("meta", attrs={"name": "title"})["content"], "· 갱신일 칸:", has_date)
try:
    if not has_date:
        raise ValueError("데이터셋 화면이 아닙니다 — 데이터셋 번호를 확인하세요")
except ValueError as e:
    print("수집 실패:", e)

## 3단계 · 13개 수집 — 그리고 스냅샷으로 숫자 고정 (수집 · 스트리밍)

2단계를 함수로 묶어 데이터셋 13개를 한 번씩 받습니다. 공공데이터포털(data.go.kr)은 화면 구조가 달라서 함수를 따로 둡니다 — 항목 이름이 `<strong class="key">` 에, 값이 그 옆 `class="value"` 에 있고, 검색 엔진용 JSON-LD(`schema.org/Dataset`)도 함께 들어 있습니다.

| 출처 | 데이터셋 | 쓴 프로젝트 |
|---|---|---|
| 서울 | OA-14991 생활인구 · OA-15182 따릉이 대여이력 · OA-13252 대여소 정보 · OA-21275 부동산 실거래가 · OA-12252 지하철 승하차 | P3 · T5 · M1 · P2 · P9 · P1 · M3 |
| 서울 | OA-22715 강우량 · OA-22824 강우량계 위치 · OA-2218 대기오염도 · OA-21285 실시간 도시데이터 · OA-15493 따릉이 실시간 | P2 · M2 · T1 |
| 포털 | 15083033 상가(상권)정보 · 15131929 고충민원 분야별 접수현황 · 15126469 아파트 매매 실거래가(OpenAPI) | P3 · T2 · P1 |

In [ ]:
SEOUL = [("OA-14991", "S"), ("OA-15182", "F"), ("OA-13252", "F"), ("OA-21275", "S"), ("OA-12252", "S"),
         ("OA-22715", "F"), ("OA-22824", "F"), ("OA-2218", "S"), ("OA-21285", "A"), ("OA-15493", "A")]
GOV = [("15083033", "fileData"), ("15131929", "fileData"), ("15126469", "openapi")]

def seoul_meta(oa, kind):
    url = f"https://data.seoul.go.kr/dataList/{oa}/{kind}/1/datasetView.do"
    s = BeautifulSoup(requests.get(url, headers=UA, timeout=30).text, "lxml")
    info = {}
    for th in s.select("th"):
        td = th.find_next_sibling("td")
        if td:
            info.setdefault(clean(th.get_text()), clean(td.get_text(" ")))
    files = []
    for tr in s.select("tr"):                                      # 파일 목록: NO · 항목 · 파일명 · 용량 · 수정일 · 내려받기
        tds = [clean(td.get_text()) for td in tr.select("td")]
        if len(tds) >= 5 and re.match(r"\d{4}\.\d\d\.\d\d\.", tds[4]):
            files.append({"name": tds[2], "mb": tds[3], "modified": tds[4]})
    faq = [{"q": clean(dl.select_one("dt").get_text()).replace(" 열기", ""), "a": clean(dl.select_one("dd").get_text())}
           for dl in s.select("#qnaList dl") if dl.select_one("dt") and dl.select_one("dd")]
    desc = s.select_one(".main-content-txt")
    return {"site": "data.seoul.go.kr", "id": oa, "url": url, "title": s.find("meta", attrs={"name": "title"})["content"],
            "published": info.get("공개일자"), "updated": info.get("데이터 갱신일"), "cycle": info.get("갱신주기"),
            "meta_modified": info.get("메타정보 수정일"), "next_due": None, "license": info.get("이용허락범위"),
            "org": info.get("제공기관"), "rows": None, "files": files,
            "desc": desc.get_text("\n").strip() if desc else "", "faq": faq}

def gov_meta(no, kind):
    url = f"https://www.data.go.kr/data/{no}/{kind}.do"
    s = BeautifulSoup(requests.get(url, headers=UA, timeout=30).text, "lxml")
    kv, desc = {}, ""
    for k in s.select("strong.key"):
        v = k.find_next_sibling(class_="value")
        if v:
            kv.setdefault(clean(k.get_text()), clean(v.get_text(" ")))
            if clean(k.get_text()) == "설명" and not desc:
                desc = v.get_text("\n").strip()
    ld = s.find("script", type="application/___③___")   # ③ 검색 엔진용 구조화 메타데이터
    j = json.loads(ld.string) if ld else {}
    return {"site": "data.go.kr", "id": no, "url": url, "title": j.get("name"),
            "published": kv.get("등록일") or j.get("datePublished"), "updated": kv.get("수정일") or j.get("dateModified"),
            "cycle": kv.get("업데이트 주기"), "meta_modified": None, "next_due": kv.get("차기 등록 예정일"),
            "license": (j.get("license") or {}).get("name") or kv.get("이용허락범위"),
            "org": kv.get("제공기관"), "rows": kv.get("전체 행"), "files": [], "desc": desc or j.get("description", ""), "faq": []}

`LIVE = True` 이면 지금 사이트에서 받고, 요청 사이에 1.5초씩 쉽니다. 받은 결과는 **형태만** 확인합니다 — 13개 모두 제목과 갱신일이 있는지, 날짜가 날짜 모양인지.

In [ ]:
LIVE = True
live = []
if LIVE:
    for oa, kind in SEOUL:
        live.append(seoul_meta(oa, kind))
        time.___④___(1.5)   # ④ 요청 사이에 쉬기
    for no, kind in GOV:
        live.append(gov_meta(no, kind))
        time.sleep(1.5)
    for m in live:
        print(f"{m['id']:9s} {m['updated']:12s} {str(m['cycle']):12s} 파일 {len(m['files']):2d}  {m['title'][:28]}")

> **확인 포인트** — 13줄이 찍히고, 모든 데이터셋에 제목과 갱신일이 있습니다(날짜 값은 받는 날에 따라 다릅니다).

In [ ]:
DATE_RE = r"\d{4}[.-]\d\d[.-]\d\d\.?"
if LIVE:
    assert len(live) == 13 and all(m["title"] and re.fullmatch(DATE_RE, m["updated"]) for m in live)

이제 숫자를 고정하기 위해 **2026-10-29 에 같은 함수로 수집해 둔 스냅샷**을 읽습니다. 이후 단계는 모두 스냅샷으로 진행합니다. 지금 받은 것과 갱신일이 달라진 데이터셋이 있으면 그 사이에 바뀐 것입니다.

In [ ]:
snap = json.load(open("data/t6_meta_20261029.json", encoding="utf-8"))
AS_OF = pd.Timestamp(snap["collected_at"][:10])
metas = snap["datasets"]
print("스냅샷 수집 시각:", snap["collected_at"], "·", len(metas), "개")
if LIVE:
    changed = [(a["id"], a["updated"], b["updated"]) for a, b in zip(metas, live) if a["updated"] != b["updated"]]
    print("스냅샷 이후 갱신일이 바뀐 데이터셋:", changed)

> **확인 포인트** — 스냅샷 13개, 기준일 2026-10-29.

In [ ]:
assert len(metas) == 13 and str(AS_OF.date()) == "2026-10-29"

## 4단계 · 메타 표 — 두 사이트의 날짜 모양 맞추기 (레이크 · 품질)

두 사이트는 날짜를 다르게 적습니다 — 서울은 `2026.08.05.`, 포털은 `2026-08-05`. 갱신 주기도 `정기(매일)` · `매일1회` · `매월 5일` · `반기` · `분기` · `년간` · `연간` · `비정기(수시)` · `실시간` 처럼 제각각입니다.
날짜는 한 모양으로, 갱신 주기는 **「며칠까지 갱신이 없어도 정상인가」(허용 일수)** 로 바꿉니다. 이 표가 다음 단계 규칙의 기준이 됩니다.

In [ ]:
def to_date(s):
    return pd.to_datetime(s.replace(".", "-").strip("-")) if s else pd.NaT

ALLOW = [("실시간", 1), ("매일", 3), ("매월", 40), ("분기", 100), ("반기", 200), ("년간", 400), ("연간", 400)]

def allow_days(cycle):
    for key, days in ALLOW:
        if cycle and key in cycle:
            return days
    return None                                   # 비정기 — 주기로는 판정하지 않습니다

meta = pd.DataFrame([{
    "id": m["id"], "site": m["site"], "title": m["title"], "cycle": m["cycle"],
    "updated": to_date(m["updated"]), "next_due": to_date(m["next_due"]),
    "newest_file": max((to_date(f["modified"]) for f in m["files"]), default=pd.NaT),
    "n_files": len(m["files"]), "license": m["license"],
} for m in metas])
meta["allow_days"] = meta["cycle"].map(allow_days)
meta["age_days"] = (AS_OF - meta["___⑤___"]).dt.days   # ⑤ 최종 갱신일 열
meta[["id", "cycle", "allow_days", "updated", "age_days", "newest_file", "n_files"]]

> **확인 포인트** — 13행. 생활인구(OA-14991)는 갱신 주기 `정기(매일)` → 허용 3일인데 경과 **85일**, 대기오염도(OA-2218)는 `비정기(수시)` 라 허용 일수가 비어 있습니다.

In [ ]:
row = meta.set_index("id")
assert len(meta) == 13 and row.loc["OA-14991", "allow_days"] == 3 and row.loc["OA-14991", "age_days"] == 85
assert pd.isna(row.loc["OA-2218", "allow_days"])

> **Aebon's Tip** — 허용 일수는 직접 정하는 **규칙**입니다. 「매일」을 3일로 둔 것은 주말 · 공휴일을 넘겨도 헛알림이 나지 않게 하려는 것입니다. 규칙 표를 코드 안에 두지 않고 이렇게 표로 꺼내 두면 담당자가 숫자만 고칠 수 있습니다.

## 5단계 · 규칙 ① 신선도 — 갱신이 멈춘 데이터 (레이크 · 품질)

세 가지를 봅니다. DuckDB 로 메타 표를 SQL 로 다룹니다(3주차 14장).

1. **주기 초과** — 경과 일수가 허용 일수를 넘었나
2. **메타 불일치** — 파일 목록의 가장 새 파일이 「데이터 갱신일」보다 사흘 넘게 늦나(파일은 바뀌었는데 갱신일 칸은 그대로). 하루 이틀 차이는 올리는 작업이 날을 넘긴 것이라 봐 줍니다
3. **곧 갱신 예정** — 포털의 「차기 등록 예정일」이 7일 안이면, 새 판이 나오면 다시 받을 준비를 합니다

In [ ]:
fresh = duckdb.sql("""
    SELECT id, title, cycle, age_days, allow_days,
           CASE WHEN age_days > allow_days THEN 'stale'
                WHEN date_diff('day', updated::DATE, newest_file::DATE) > 3 THEN 'meta_mismatch'
                WHEN next_due IS NOT NULL AND date_diff('day', DATE '2026-10-29', next_due::DATE) ___⑥___ 0 AND 7 THEN 'due_soon'   -- ⑥ 두 값 사이
                ELSE 'ok' END AS status
    FROM meta
    ORDER BY status, id
""").df()
fresh[fresh["status"] != "ok"]

> **확인 포인트** — `ok` 가 아닌 것은 넷입니다.
> **stale** — 생활인구(OA-14991, 매일인데 85일)와 국토교통부 아파트 매매 실거래가 OpenAPI(15126469, 실시간인데 99일) · **meta_mismatch** — 대기오염도(OA-2218, 갱신일 2025-11-04 인데 파일은 2026-04-01) · **due_soon** — 상가(상권)정보(15083033, 차기 등록 예정일 2026-10-31).

In [ ]:
bad = fresh.set_index("id")["status"]
assert bad[bad != "ok"].to_dict() == {"15083033": "due_soon", "OA-2218": "meta_mismatch", "15126469": "stale", "OA-14991": "stale"}

그런데 넷 중 하나는 **헛알림**입니다. 15126469 는 OpenAPI 라서 포털 화면의 「수정일」은 API 설명 화면을 고친 날이지 자료가 들어온 날이 아닙니다(실제로는 신고가 들어올 때마다 바로 조회됩니다).
규칙을 고쳐 봅니다 — OpenAPI(실시간) 데이터셋은 화면 메타데이터가 아니라 **API 를 직접 불러** 최근 자료가 오는지로 판정해야 합니다(T1 의 실시간 수집처럼). 여기서는 판정 대상에서 빼고 `check_by_api` 로 표시합니다.

In [ ]:
meta["check"] = meta.apply(lambda r: "check_by_api" if r["cycle"] == "실시간" else "meta", axis=1)
fresh = fresh.merge(meta[["id", "check"]], on="id")
fresh.loc[fresh["check"] == "check_by_api", "status"] = "check_by_api"
print(fresh["status"].value_counts().to_dict())

> **확인 포인트** — `ok` 7 · `check_by_api` 3(실시간 도시데이터 · 따릉이 실시간 · 아파트 실거래가 API) · `stale` · `meta_mismatch` · `due_soon` 각 1.

In [ ]:
assert fresh["status"].value_counts().to_dict() == {"ok": 7, "check_by_api": 3, "stale": 1, "meta_mismatch": 1, "due_soon": 1}

갱신일 칸 하나만 믿지 않도록 **파일 이름**도 봅니다. 생활인구는 달마다 `LOCAL_PEOPLE_DONG_YYYYMM.zip` 을 올립니다. 이름에서 달을 꺼내면 「마지막 파일이 몇 달 전 것인가」가 나옵니다.

In [ ]:
pop_files = next(m for m in metas if m["id"] == "OA-14991")["files"]
months = sorted(re.search(r"(\d{6})\.zip", f["name"]).group(1) for f in pop_files if re.search(r"_(\d{6})\.zip", f["name"]))
last = pd.Period(months[-1], "M")
lag = (AS_OF.to_period("M") - last).___⑦___   # ⑦ 두 달 사이의 개월 수
print("월별 파일", len(months), "개 ·", months[0], "~", months[-1], "· 기준일까지", lag, "개월")

> **확인 포인트** — 월별 파일 **43개(202301 ~ 202607)**, 마지막 파일이 기준일(10월)보다 3개월 앞입니다. 매달 5일 무렵 지난달 파일이 올라오던 데이터셋이니 9월분까지 있어야 정상입니다.

In [ ]:
assert (len(months), months[0], months[-1], lag) == (43, "202301", "202607", 3)

## 6단계 · 규칙 ② 형식 지문 — 열 이름이 같아도 바뀐 것 (레이크 · 품질)

같은 데이터셋의 해마다 올라오는 파일이 늘 같은 모양일까요? 대기오염도(OA-2218)의 2023 · 2024 · 2025년 파일로 확인합니다(M2 가 쓴 데이터).
파일마다 **지문**(fingerprint)을 만듭니다 — 인코딩, 머리줄에 따옴표가 있는지, 열 이름 목록, 행 수. 지문이 앞 판과 다르면 파이프라인의 읽기 코드가 깨질 수 있다는 신호입니다.

In [ ]:
def sniff(raw):
    """바이트 → (인코딩, 글자). BOM 이 있으면 utf-8-sig, 없으면 utf-8 을 먼저 해 보고 안 되면 cp949."""
    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig", raw.decode("utf-8-sig")
    try:
        return "utf-8", raw.decode("utf-8")
    except UnicodeDecodeError:
        return "___⑧___", raw.decode("cp949")   # ⑧ 한글 윈도 인코딩

def fingerprint(path):
    raw = gzip.open(path).read() if path.endswith(".gz") else open(path, "rb").read()
    enc, text = sniff(raw)
    head = text.splitlines()[0]
    df = pd.read_csv(io.StringIO(text), dtype=str)
    return {"file": os.path.basename(path), "encoding": enc, "quoted": head.___⑨___('"'),   # ⑨ 첫 글자 확인
            "columns": "|".join(df.columns), "n_cols": df.shape[1], "rows": len(df)}

AIR = ["t6_seoul_air_daily_2023.csv.gz", "t6_seoul_air_daily_2024.csv.gz", "seoul_air_daily_2025.csv.gz"]
fp = pd.DataFrame([fingerprint(os.path.join("data", f)) for f in AIR])
fp[["file", "encoding", "quoted", "n_cols", "rows"]]

> **확인 포인트** — 열 8개 · 열 이름은 세 해가 같습니다. 그런데 **인코딩이 해마다 다릅니다** — 2023 CP949(따옴표 없음) → 2024 UTF-8 BOM(따옴표) → 2025 CP949(따옴표). 행 수는 2023 18,250 · 2024 18,300(윤년 366일 × 50곳) · 2025 18,250 입니다.

In [ ]:
assert fp["columns"].nunique() == 1 and fp["n_cols"].tolist() == [8, 8, 8]
assert fp["encoding"].tolist() == ["cp949", "utf-8-sig", "cp949"] and fp["quoted"].tolist() == [False, True, True]
assert fp["rows"].tolist() == [18250, 18300, 18250]

M2 처럼 「이 데이터셋은 CP949」라고 정해 두고 읽으면 2024년 파일에서 깨집니다. 일부러 해 보세요.

In [ ]:
try:
    x = pd.read_csv(os.path.join("data", AIR[1]), encoding="cp949", nrows=3)
    print("오류 없이 읽혔지만 열 이름이 깨짐:", list(x.columns)[:3])
except UnicodeDecodeError as e:
    print("2024년 파일 읽기 실패:", str(e)[:70])

이웃한 판끼리 지문을 비교해 바뀐 항목을 알림으로 만듭니다. 같은 방법으로 따릉이 대여소 정보(OA-13252)의 네 시점 xlsx 도 봅니다 — xlsx 는 인코딩이 없으니 머리글 5줄과 행 수를 지문으로 씁니다.

In [ ]:
def diff_fp(df, keys):
    out = []
    for a, b in zip(df.to_dict("records"), df.to_dict("records")[1:]):
        for k in keys:
            if a[k] != b[k]:
                out.append({"from": a["file"], "to": b["file"], "what": k, "before": a[k], "after": b[k]})
    return pd.DataFrame(out)

STATIONS = ["p9_bike_station_202412.xlsx", "p9_bike_station_202506.xlsx", "p9_bike_station_202512.xlsx", "p9_bike_station_202606.xlsx"]
def xlsx_fp(path):
    x = pd.read_excel(path, header=None, dtype=str)
    head = "|".join(x.iloc[:5].fillna("").astype(str).agg(" ".join))
    return {"file": os.path.basename(path), "header": head, "rows": len(x) - 5}

st_fp = pd.DataFrame([xlsx_fp(os.path.join("data", f)) for f in STATIONS])
air_diff = diff_fp(fp, ["encoding", "quoted", "columns"])
st_diff = diff_fp(st_fp, ["header"])
print(air_diff[["to", "what", "before", "after"]].to_string(index=False))
print("대여소 머리글 변화:", len(st_diff), "· 대여소 수:", st_fp["rows"].tolist())

> **확인 포인트** — 대기오염도는 형식 변화 **3건**(2024년: 인코딩 · 따옴표, 2025년: 인코딩). 대여소 정보는 머리글이 네 시점 모두 같고, 대여소 수만 2,766 → 2,780 → 2,799 → 2,789 로 바뀝니다.

In [ ]:
assert len(air_diff) == 3 and len(st_diff) == 0 and st_fp["rows"].tolist() == [2766, 2780, 2799, 2789]

## 7단계 · 규칙 ③ 품질 지표 — 빈 값과 행 수 변화 (레이크 · 품질)

이번에는 파일 **안**을 봅니다. 앞 프로젝트들의 스냅샷 셋에 대해 열마다 빈 값 비율을 DuckDB 로 셉니다. `SUMMARIZE` 는 열마다 개수 · 빈 값 비율(`null_percentage`)을 한 번에 냅니다(3주차 14장).

In [ ]:
trade = pd.read_csv("data/seoul_realestate_trade_202606_09.csv.gz", encoding="cp949", dtype=str)
pop = pd.read_csv("data/seoul_living_pop_dong_202606.csv.gz", dtype=str, index_col=False)     # 행 끝 쉼표 — P3 2단계
acrc = pd.read_csv("data/acrc_complaints_by_field_2015_2025.csv", encoding="cp949", dtype=str)

nulls = []
for name, df in [("trade", trade), ("living_pop", pop), ("acrc", acrc)]:
    s = duckdb.sql("___⑩___ SELECT * FROM df").df()   # ⑩ 열마다 요약 통계
    for _, r in s.iterrows():
        nulls.append({"dataset": name, "column": r["column_name"], "null_pct": round(float(r["null_percentage"]), 1)})
nulls = pd.DataFrame(nulls)
nulls[nulls["null_pct"] > 0].sort_values("null_pct", ascending=False)

> **확인 포인트** — 빈 값이 50% 를 넘는 열은 실거래가의 `권리구분`(98.7%) · `취소일`(96.8%) 둘입니다. 생활인구 · 고충민원은 빈 값이 없습니다.

In [ ]:
over = nulls[nulls["null_pct"] > 50].set_index("column")["null_pct"].to_dict()
assert over == {"권리구분": 98.7, "취소일": 96.8}
assert nulls[nulls["dataset"] != "trade"]["null_pct"].max() == 0

`취소일` 은 해제 신고가 있을 때만 채워지는 열이라 **비어 있는 것이 정상**입니다(P1 3단계). 빈 값 규칙을 그대로 두면 날마다 같은 헛알림이 납니다. 「비어야 정상인 열」 목록을 따로 두고, 그 열은 거꾸로 **갑자기 채워지면** 알립니다.

In [ ]:
EXPECTED_EMPTY = {("trade", "권리구분"), ("trade", "취소일")}
nulls["rule"] = nulls.apply(lambda r: "expected_empty" if (r["dataset"], r["column"]) in EXPECTED_EMPTY
                            else ("too_empty" if r["null_pct"] > 50 else "ok"), axis=1)
print(nulls["rule"].value_counts().to_dict())

다음은 **기간별 행 수**입니다. 실거래가는 계약 달마다, 생활인구는 날짜마다 셉니다. 앞 기간보다 20% 넘게 줄면 알립니다 — 윈도 함수 `lag()` 로 바로 앞 기간 값을 옆에 붙입니다(2주차 8장).

In [ ]:
counts = pd.concat([
    trade.assign(period=trade["계약일"].str[:6]).groupby("period").size().rename("rows").reset_index().assign(dataset="trade"),
    pop.groupby("기준일ID").size().rename("rows").reset_index().rename(columns={"기준일ID": "period"}).assign(dataset="living_pop"),
])
change = duckdb.sql("""
    SELECT dataset, period, rows, prev,
           round(100.0 * (rows - prev) / prev, 1) AS pct
    FROM (
        SELECT dataset, period, rows,
               ___⑪___(rows) OVER (PARTITION BY dataset ORDER BY period) AS prev   -- ⑪ 바로 앞 행의 값
        FROM counts
    )
    WHERE prev IS NOT NULL
    ORDER BY dataset, period
""").df()
print(change[change["dataset"] == "trade"].to_string(index=False))
print("생활인구 날짜별 행 수:", sorted(set(change[change["dataset"] == "living_pop"]["rows"])))

> **확인 포인트** — 실거래가는 6월 10,348 · 7월 10,531 · **8월 7,488(-28.9%)** · 9월 7,243. 생활인구는 30일 모두 하루 10,176행(424개 동 × 24시간)으로 일정합니다.

In [ ]:
t = change[change["dataset"] == "trade"].set_index("period")
assert t["rows"].to_dict() == {"202607": 10531, "202608": 7488, "202609": 7243} and t.loc["202608", "pct"] == -28.9
assert sorted(set(change[change["dataset"] == "living_pop"]["rows"])) == [10176]

8월 · 9월이 줄어든 것은 거래가 줄어서가 아니라 **아직 신고가 덜 들어와서**일 가능성이 큽니다 — 계약 후 30일 안에 신고하고, 스냅샷은 10월 28일에 받았습니다. 이런 「최근 기간 미완성」은 알림을 내되 원인 후보를 함께 적어 둡니다.

마지막으로, 포털이 화면에 적어 둔 「전체 행」과 실제 파일의 행 수를 맞춰 봅니다. 고충민원(15131929)은 포털에 286행이라고 적혀 있습니다.

In [ ]:
portal_rows = int(next(m for m in metas if m["id"] == "15131929")["rows"])
print("포털 286?", portal_rows, "· 실제 파일:", len(acrc))
assert portal_rows == len(acrc) == 286

## 8단계 · 알림 목록과 관제판 (RAG · 서비스)

세 규칙의 결과를 **알림 하나 = 한 행**으로 모읍니다. 등급은 셋입니다 — `error`(지금 파이프라인이 틀린 값을 낼 수 있음) · `warn`(확인 필요) · `info`(알아 둘 것).

In [ ]:
alerts = []
for _, r in fresh[fresh["status"].isin(["stale", "meta_mismatch", "due_soon"])].iterrows():
    sev = {"stale": "error", "meta_mismatch": "warn", "due_soon": "info"}[r["status"]]
    alerts.append({"dataset": r["id"], "rule": "freshness:" + r["status"], "severity": sev,
                   "detail": f"{r['cycle']} · 경과 {r['age_days']}일"})
for _, r in air_diff.iterrows():
    alerts.append({"dataset": "OA-2218", "rule": "format:" + r["what"], "severity": "error",
                   "detail": f"{r['to']}: {r['before']} → {r['after']}"})
for _, r in change[change["pct"] < -20].iterrows():
    alerts.append({"dataset": "OA-21275", "rule": "rows:drop", "severity": "warn",
                   "detail": f"{r['period']} {r['prev']:,} → {r['rows']:,} ({r['pct']}%) · 최근 기간 신고 미완성 가능"})
for _, r in nulls[nulls["rule"] == "too_empty"].iterrows():
    alerts.append({"dataset": r["dataset"], "rule": "nulls:too_empty", "severity": "warn", "detail": f"{r['column']} {r['null_pct']}%"})
alerts = pd.DataFrame(alerts)
os.makedirs("out", exist_ok=True)
alerts.___⑫___("out/alerts.csv", index=False)   # ⑫ 표를 CSV 파일로
meta.to_csv("out/meta.csv", index=False)
alerts

> **확인 포인트** — 알림 **7건** — error 4(생활인구 stale · 대기오염도 형식 3) · warn 2(대기오염도 메타 불일치 · 실거래가 8월 행 수 급감) · info 1(상가정보 갱신 예정).

In [ ]:
assert len(alerts) == 7 and alerts["severity"].value_counts().to_dict() == {"error": 4, "warn": 2, "info": 1}

노트북에서는 matplotlib 으로 같은 그림을 그립니다 — 데이터셋마다 「경과 일수 ÷ 허용 일수」(1 을 넘으면 늦음)입니다.

In [ ]:
import matplotlib.pyplot as plt

v = meta.dropna(subset=["allow_days"]).assign(ratio=lambda d: d["age_days"] / d["allow_days"]).sort_values("ratio")
v = v[v["cycle"] != "실시간"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(v["id"], v["ratio"], color=["#c0392b" if x > 1 else "#7f8c8d" for x in v["ratio"]])
ax.axvline(1, ls="--", c="k", lw=1)
ax.set_xlabel("days since last update / allowed days")
ax.set_title("Dataset freshness (as of 2026-10-29)")
plt.tight_layout()
plt.show()
print(v[["id", "ratio"]].round(2).to_string(index=False))

> **확인 포인트** — 막대가 기준선(1)을 넘는 것은 OA-14991 하나(28.33)입니다.

In [ ]:
assert v[v["ratio"] > 1]["id"].tolist() == ["OA-14991"]

관제판은 Streamlit 앱 하나로 만듭니다(8주차). `out/` 의 두 표만 읽으므로 수집 · 규칙과 화면이 분리됩니다.

In [ ]:
%%writefile app.py
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Public data quality board", layout="wide")
alerts = pd.read_csv("out/alerts.csv")
meta = pd.read_csv("out/meta.csv")

st.title("공공데이터 품질 관제판")
c1, c2, c3 = st.columns(3)
c1.metric("감시 데이터셋", len(meta))
c2.metric("error", int((alerts["severity"] == "error").sum()))
c3.metric("warn", int((alerts["severity"] == "warn").sum()))
sev = st.multiselect("등급", ["error", "warn", "info"], default=["error", "warn"])
st.dataframe(alerts[alerts["severity"].isin(sev)], use_container_width=True)
ds = st.selectbox("데이터셋", meta["id"].tolist())
st.write(meta[meta["id"] == ds].T)

Colab 에서는 서버를 띄우지 않고 `AppTest` 로 앱을 한 번 돌려 오류가 없는지 검사합니다(P1 9단계와 같은 방법).

In [ ]:
import logging
from streamlit.testing.v1 import AppTest
logging.getLogger("streamlit").setLevel(logging.ERROR)

at = AppTest.from_file(os.path.abspath("app.py"), default_timeout=60).run()
print("오류:", list(at.exception))
print("지표:", [(m.label, m.value) for m in at.metric])
print("표의 행:", len(at.dataframe[0].value))

> **확인 포인트** — 오류 없이 지표 `감시 데이터셋 13 · error 4 · warn 2`, 기본 선택(error · warn)으로 표에 6행이 나옵니다.

In [ ]:
assert len(at.exception) == 0 and [m.value for m in at.metric] == ["13", "4", "2"]
assert len(at.dataframe[0].value) == 6

등급에서 `info` 를 더하면 7행이 되는지 확인합니다.

In [ ]:
at.multiselect[0].set_value(["error", "warn", "info"]).run()
assert len(at.dataframe[0].value) == 7

## 9단계 · 「이 데이터 왜 끊겼나」 — 공지 검색 (RAG · 서비스)

알림은 「무엇이」 이상한지만 말합니다. **왜** 그런지는 사람이 쓴 글에 있습니다 — 데이터셋 설명의 ※ 안내, 「[데이터 변경 안내]」, 자주 하는 질문의 답.
그 글을 문장 단위 문서로 잘라 Qdrant 에 넣고(7주차 청킹), 알림이 뜬 데이터셋의 이유를 검색합니다.

In [ ]:
docs = []
for m in metas:
    for line in m["desc"].splitlines():
        line = clean(line)
        if len(line) >= 15:
            docs.append({"id": m["id"], "title": m["title"], "kind": "desc", "text": line})
    for q in m["faq"]:
        docs.append({"id": m["id"], "title": m["title"], "kind": "faq", "text": f"질문: {q['q']} 답: {q['a']}"})
print(len(docs), "개 문서 ·", pd.Series([d["kind"] for d in docs]).value_counts().to_dict())

> **확인 포인트** — 문서 45개(설명 문장 40 · 자주 하는 질문 5).

In [ ]:
assert len(docs) == 45 and sum(d["kind"] == "faq" for d in docs) == 5

임베딩은 키 없이 도는 한국어 모델 `jhgan/ko-sroberta-multitask`(768차원), 벡터 DB 는 메모리 Qdrant 입니다. 문서 앞에 데이터셋 제목을 붙여 넣어, 「생활인구」 같은 낱말이 본문에 없어도 그 데이터셋의 문장이 찾아지게 합니다.

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

emb = SentenceTransformer("jhgan/ko-sroberta-multitask")
qd = QdrantClient(":memory:")
qd.create_collection("t6_notice", vectors_config=VectorParams(size=768, distance=Distance.___⑬___))   # ⑬ 방향이 비슷한 정도
vecs = emb.encode([f"{d['title']} — {d['text']}" for d in docs], normalize_embeddings=True)
qd.upsert("t6_notice", points=[PointStruct(id=i, vector=v.tolist(), payload=d) for i, (d, v) in enumerate(zip(docs, vecs))])

def search(q, k=3, dataset=None):
    flt = Filter(must=[FieldCondition(key="id", match=MatchValue(value=dataset))]) if dataset else None
    return qd.query_points("t6_notice", query=emb.encode(q, normalize_embeddings=True).tolist(),
                           limit=k, ___⑭___=flt).points   # ⑭ 검색 범위를 좁히는 조건

for h in search("생활인구 데이터가 7월 이후로 왜 끊겼나요?"):
    print(f"{h.score:.3f} [{h.payload['id']}] {h.payload['text'][:70]}")

> **확인 포인트** — 1위가 생활인구(OA-14991)의 안내 「※ 국가표준격자(250m 격자) 단위로의 데이터 전환으로 기존 데이터 생산이 2026년 7월 31일 이후 종료됨」입니다.

In [ ]:
top = search("생활인구 데이터가 7월 이후로 왜 끊겼나요?")[0].payload
assert top["id"] == "OA-14991" and "2026년 7월 31일 이후 종료" in top["text"]

다른 질문도 해 봅니다. T5 가 부딪힌 「항동이 없다」와, 상가정보의 업종 분류가 바뀌었는지입니다.

In [ ]:
for q in ["생활인구에 항동이 없는 이유는?", "상가정보 업종 분류 체계가 바뀌었나요?"]:
    h = search(q, k=1)[0]
    print(q, "→", f"[{h.payload['id']} · {h.payload['kind']}]", h.payload["text"][:80])

> **확인 포인트** — 항동 질문은 생활인구의 자주 하는 질문(2016년 집계구 · 행정동 코드 사용), 업종 질문은 상가정보(15083033)의 [데이터 변경 안내] 가운데 「업종분류 정제에 따라 상권업종분류 연계표와 상이할 수 있음」이 1위입니다.

In [ ]:
h1, h2 = search("생활인구에 항동이 없는 이유는?", k=1)[0].payload, search("상가정보 업종 분류 체계가 바뀌었나요?", k=1)[0].payload
assert h1["id"] == "OA-14991" and h1["kind"] == "faq" and "2016년" in h1["text"]
assert h2["id"] == "15083033" and "업종분류" in h2["text"]

이제 알림과 검색을 잇습니다. `error` · `warn` 알림이 뜬 데이터셋마다, 그 데이터셋 문서 안에서만 「중단 · 종료 · 변경 · 전환」을 찾아 **원인 후보** 한 줄을 붙입니다.

In [ ]:
def reason(ds):
    hits = search("데이터 제공 중단 종료 변경 전환 안내", k=1, dataset=ds)
    return (round(hits[0].score, 3), hits[0].payload["text"]) if hits else (0, "(관련 공지 없음 — 담당 부서에 문의)")

flagged = alerts[alerts["severity"].isin(["error", "warn"])]["dataset"].unique()
why = {ds: reason(ds) for ds in flagged}
for ds, (score, t) in why.items():
    print(f"{ds:10s} {score:.3f} {t[:70]}")

> **확인 포인트** — OA-14991 에는 250m 격자 전환 안내가 붙습니다. 그런데 OA-2218(「Sheet 서비스는 최근 1년 이내의 데이터만 출력」) · OA-21275(「신고년도로 필터선택 후 내려받기」)에 붙은 문장은 형식 변화 · 행 수 급감과 **상관이 없습니다.**
> 검색은 언제나 「가장 가까운 문장」을 돌려줄 뿐, 원인이 공지에 없으면 없다고 말하지 않습니다. 점수가 가장 높은 OA-14991 만 원인으로 믿을 만합니다 — 점수 문턱을 정하거나, 원인 후보를 사람이 확인하는 칸으로 남겨 두세요.

In [ ]:
assert "250m" in why["OA-14991"][1]
assert why["OA-14991"][0] == max(s for s, _ in why.values())

일부러 데이터셋 이름을 틀리게 걸러 보세요(`OA-1499`). 오류는 나지 않고 **빈 결과**가 돌아옵니다 — 필터는 조용히 실패하므로, 빈 결과를 「공지 없음」으로 바로 믿지 말고 데이터셋 번호부터 확인하세요.

In [ ]:
print("틀린 번호로 거른 결과:", len(search("종료", dataset="OA-1499")))
try:
    qd.query_points("t6_notice", query=[0.0] * 10, limit=1)
except Exception as e:
    print("검색 실패:", type(e).__name__)

답 문장은 OpenAI `gpt-4o-mini` 가 만듭니다(7주차). **키가 있을 때만** 부르고, 없으면 근거만 보여 주는 것으로 끝납니다.

In [ ]:
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")
try:
    from google.colab import userdata                  # Colab 보안 비밀에 넣어 두었다면
    OPENAI_KEY = OPENAI_KEY or userdata.get("OPENAI_API_KEY")
except Exception:
    pass

Q = "생활인구 데이터가 7월 이후로 왜 끊겼나요?"
hits = search(Q, k=3)
context = "\n".join(f"[문서 {i}] ({h.payload['title']}) {h.payload['text']}" for i, h in enumerate(hits, 1))
if OPENAI_KEY:
    need("openai")
    from openai import OpenAI
    res = OpenAI(api_key=OPENAI_KEY).chat.completions.create(model="gpt-4o-mini", temperature=0, messages=[
        {"role": "system", "content": "공공데이터 관제 담당자입니다. 아래 근거만으로 답하고, 없으면 모른다고 하세요. 문장 끝에 [문서 n] 을 붙이세요."},
        {"role": "user", "content": context + "\n\n질문: " + Q}])
    print(res.choices[0].message.content)
else:
    print("OPENAI_API_KEY 가 없어 근거만 보여 줍니다.\n")
    print(context)

## 10단계 · Airflow DAG — 매일 수집 → 규칙 → 색인 (오케스트레이션)

관제는 매일 돌아야 의미가 있습니다. 2~9단계를 함수 셋으로 나눠 `t6_tasks.py` 에 두고, DAG 가 매일 아침 차례로 부르게 합니다(8주차).
태스크 사이에는 **파일**로 넘깁니다 — 수집 결과는 `lake/meta/dt=날짜.json`, 알림은 `lake/alerts/dt=날짜.csv`. 날짜별 파일이 쌓이면 「어제와 오늘」을 비교해 **새로 생긴 알림만** 보낼 수 있습니다.

In [ ]:
%%writefile t6_tasks.py
"""T6 공공데이터 품질 관제판 — Airflow 태스크가 부르는 함수(노트북 3 · 4~5 · 9단계를 묶은 것)."""
import json, os
import pandas as pd

LAKE = os.environ.get("T6_LAKE", "lake")
ALLOW = [("실시간", 1), ("매일", 3), ("매월", 40), ("분기", 100), ("반기", 200), ("년간", 400), ("연간", 400)]


def collect(day, live=False, snapshot="data/t6_meta_20261029.json", fetchers=None):
    """① 수집 — live 면 fetchers(목록 · 함수)로 받고, 아니면 스냅샷을 그날 파일로 둡니다."""
    metas = [f(*a) for f, a in fetchers] if live else json.load(open(snapshot, encoding="utf-8"))["datasets"]
    os.makedirs(f"{LAKE}/meta", exist_ok=True)
    json.dump(metas, open(f"{LAKE}/meta/dt={day}.json", "w", encoding="utf-8"), ensure_ascii=False)
    return len(metas)


def check_freshness(day):
    """② 규칙 — 그날 메타로 신선도 알림을 만들어 lake/alerts/dt=날짜.csv 로."""
    metas = json.load(open(f"{LAKE}/meta/dt={day}.json", encoding="utf-8"))
    as_of = pd.Timestamp(day)
    rows = []
    for m in metas:
        allow = next((d for k, d in ALLOW if m["cycle"] and k in m["cycle"]), None)
        upd = pd.to_datetime(m["updated"].replace(".", "-").strip("-"))
        age = (as_of - upd).days
        if m["cycle"] != "실시간" and allow and age > allow:
            rows.append({"dataset": m["id"], "rule": "freshness:stale", "severity": "error", "detail": f"{m['cycle']} · 경과 {age}일"})
    os.makedirs(f"{LAKE}/alerts", exist_ok=True)
    pd.DataFrame(rows, columns=["dataset", "rule", "severity", "detail"]).to_csv(f"{LAKE}/alerts/dt={day}.csv", index=False)
    return len(rows)


def new_alerts(day, prev_day):
    """③ 알림 — 어제 없던 알림만 돌려줍니다(같은 알림을 날마다 다시 보내지 않게)."""
    today = pd.read_csv(f"{LAKE}/alerts/dt={day}.csv")
    if not os.path.exists(f"{LAKE}/alerts/dt={prev_day}.csv"):
        return today
    before = pd.read_csv(f"{LAKE}/alerts/dt={prev_day}.csv")
    key = ["dataset", "rule"]
    return today.merge(before[key], on=key, how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge")

In [ ]:
os.makedirs("dags", exist_ok=True)

In [ ]:
%%writefile dags/t6_quality_daily.py
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

import t6_tasks as T

with DAG(
    dag_id="t6_quality_daily",
    start_date=datetime(2026, 11, 1),
    schedule="___⑮___",   # ⑮ 매일 아침 7시(크론)
    catchup=False,
    default_args={"retries": 2, "retry_delay": timedelta(minutes=10)},
    tags=["T6", "quality"],
) as dag:
    def _collect(ds, **_):
        return T.collect(ds)

    def _check(ds, **_):
        return T.check_freshness(ds)

    def _notify(ds, prev_ds, **_):
        return len(T.new_alerts(ds, prev_ds))

    collect = PythonOperator(task_id="collect_meta", python_callable=_collect)
    check = PythonOperator(task_id="check_freshness", python_callable=_check)
    notify = PythonOperator(task_id="new_alerts", python_callable=_notify)
    collect >> check >> notify

`ds` · `prev_ds` 는 Airflow 가 태스크 함수에 넘겨 주는 실행 날짜와 그 전 실행 날짜(`2026-11-01` 꼴)입니다(8주차). 노트북에서는 그 날짜를 직접 넣어 이틀치를 차례로 돌려 봅니다 — 둘째 날에는 **새 알림이 없어야** 합니다.

In [ ]:
import importlib
os.environ["T6_LAKE"] = "lake"
shutil.rmtree("lake", ignore_errors=True)
import t6_tasks as T
importlib.reload(T)

for day in ["2026-10-29", "2026-10-30"]:
    n = T.collect(day)
    k = T.check_freshness(day)
    print(day, "· 메타", n, "· 알림", k)
first = T.new_alerts("2026-10-29", "2026-10-28")
second = T.___⑯___("2026-10-30", "2026-10-29")   # ⑯ 어제 없던 알림만
print("첫날 새 알림:", first["dataset"].tolist(), "· 둘째 날 새 알림:", second["dataset"].tolist())

> **확인 포인트** — 이틀 모두 알림 1건(OA-14991 stale)이고, 첫날에는 새 알림 1건, 둘째 날에는 0건입니다 — 같은 알림을 날마다 다시 보내지 않습니다.

In [ ]:
assert first["dataset"].tolist() == ["OA-14991"] and len(second) == 0
assert sorted(os.listdir("lake/meta")) == ["dt=2026-10-29.json", "dt=2026-10-30.json"]

일부러 순서를 어겨 보세요. 수집 전에 규칙을 부르면 그날 메타 파일이 없어 실패합니다 — DAG 의 `>>` 가 막아 주는 일입니다.

In [ ]:
try:
    T.check_freshness("2026-10-31")
except FileNotFoundError as e:
    print("check_freshness 실패:", type(e).__name__, os.path.basename(e.filename))

> **Aebon's Tip** — 8주차 로컬 Airflow 로 옮길 때는 `dags/` 에 `t6_quality_daily.py` 와 `t6_tasks.py` 를 함께 넣고, `collect` 의 `live=True` 와 `fetchers`(3단계의 `seoul_meta` · `gov_meta` 와 인자 목록)를 넘기세요. 요청 간격(1.5초)은 `fetchers` 안의 함수에서 지킵니다.
> 새 알림은 `new_alerts` 다음에 이메일 · 슬랙 태스크를 붙여 보냅니다(8주차 알림).

## 11단계 · 더 해 보기 · 발표 준비

- 수집 · 스트리밍: 감시 목록을 이 과정의 모든 프로젝트 데이터셋으로 늘리세요(국가법령정보 · 정책브리핑 등). 사이트마다 화면 구조가 다르니 `seoul_meta` · `gov_meta` 같은 함수를 하나씩 더 만듭니다.
- 레이크 · 품질: 6단계 지문에 「열마다 값 모양」(날짜가 `20250101` 인지 `2025-01-01` 인지, 숫자에 쉼표가 있는지)을 더하세요. 7단계 행 수 규칙을 「최근 2개월은 신고 미완성」으로 다듬어 헛알림을 줄여 보세요.
- RAG · 서비스: 9단계의 원인 후보를 관제판(app.py)의 알림 표 옆 칸으로 붙이세요. 지금 받은 화면(LIVE)과 스냅샷의 설명이 달라진 문장만 뽑으면 「새로 올라온 공지」가 됩니다.
- 오케스트레이션: `check_freshness` 에 6 · 7단계 규칙도 넣고, 한 데이터셋 화면이 응답하지 않을 때 그 데이터셋만 「수집 실패」 알림으로 남기고 나머지는 계속 도는지 확인하세요.

## 정답

① `th` — 표의 머리 칸 태그
② `find_next_sibling` — 바로 옆 형제 태그
③ `ld+json` — 검색 엔진용 구조화 메타데이터
④ `sleep` — 요청 사이에 쉬기
⑤ `updated` — 최종 갱신일 열
⑥ `BETWEEN` — 두 값 사이
⑦ `n` — 두 달 사이의 개월 수
⑧ `cp949` — 한글 윈도 인코딩
⑨ `startswith` — 첫 글자 확인
⑩ `SUMMARIZE` — 열마다 요약 통계
⑪ `lag` — 바로 앞 행의 값
⑫ `to_csv` — 표를 CSV 파일로
⑬ `COSINE` — 방향이 비슷한 정도
⑭ `query_filter` — 검색 범위를 좁히는 조건
⑮ `0 7 * * *` — 매일 아침 7시(크론)
⑯ `new_alerts` — 어제 없던 알림만